# Chapter 17: Shape, Skewness, and Outliers

Fixed-seed synthetic NRG data illustrate shape and outlier screening. A flag is a prompt to investigate, not an instruction to delete.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from datasciencebook.shape import bowley_skewness, iqr_fences, iqr_flags, moment_skewness, robust_z_scores

rng = np.random.default_rng(20260829)
symmetric = rng.normal(5, 1, 300)
right_skewed = rng.lognormal(0.9, 0.55, 300)
bimodal = np.r_[rng.normal(3, 0.45, 150), rng.normal(7, 0.6, 150)]
print("Samples created: 300 observations each")

Samples created: 300 observations each


In [2]:
for name, values in [("Symmetric", symmetric), ("Right-skewed", right_skewed), ("Bimodal", bimodal)]:
    print(f"{name}: moment={moment_skewness(values):.3f}, Bowley={bowley_skewness(values):.3f}")

Symmetric: moment=0.037, Bowley=-0.049
Right-skewed: moment=1.598, Bowley=0.175
Bimodal: moment=0.046, Bowley=0.021


In [3]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for axis, name, values in zip(axes, ["Approximately symmetric", "Right-skewed", "Bimodal mixture"], [symmetric, right_skewed, bimodal]):
    axis.hist(values, bins=22, color="#287271", edgecolor="white")
    axis.set(title=name, xlabel="Value", ylabel="Count")
fig.tight_layout()
plt.show()

<Figure size 1200x350 with 3 Axes>

In [4]:
lower, upper = iqr_fences(right_skewed)
flags = iqr_flags(right_skewed)
print(f"IQR fences: {lower:.3f} to {upper:.3f}")
print(f"Flagged observations: {sum(flags)} of {len(flags)}")

IQR fences: -1.250 to 6.643
Flagged observations: 12 of 300


In [5]:
with_disruption = np.r_[right_skewed, 30.0]
scores = robust_z_scores(with_disruption)
print(f"With 30-day event: moment skewness={moment_skewness(with_disruption):.3f}")
print(f"With 30-day event: Bowley skewness={bowley_skewness(with_disruption):.3f}")
print(f"Maximum robust score={max(scores):.3f}")

With 30-day event: moment skewness=6.250
With 30-day event: Bowley skewness=0.173
Maximum robust score=19.918


In [6]:
fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4))
axes2[0].boxplot(right_skewed, vert=False)
axes2[0].set(title="Conventional boxplot flags long-tail values", xlabel="Delivery days")
axes2[1].scatter(np.arange(len(with_disruption)), scores, s=12, color="#287271")
axes2[1].axhline(3.5, color="#d1495b", linestyle="--", label="Screening threshold")
axes2[1].set(title="Median/MAD robust scores", xlabel="Observation", ylabel="Robust score")
axes2[1].legend()
fig2.tight_layout()
plt.show()

<Figure size 1000x400 with 2 Axes>

In [7]:
verified_as_real = True
decision = "retain, label disruption, and report sensitivity" if verified_as_real else "correct from source or mark missing"
print(f"Decision: {decision}.")

Decision: retain, label disruption, and report sensitivity.


## Learner practice
Create a grouped shape analysis for delivery routes. Explain whether each flagged value appears erroneous, rare but valid, or generated by a different process.